# 01 — Data Processing Pipeline

This notebook converts raw GDC files into one modeling table. It assumes the raw data exists under `data/raw/` using the folder structure described in the README.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root(start: Path | None = None) -> Path:
    if "TCGA_TAA_PROJECT_ROOT" in os.environ:
        return Path(os.environ["TCGA_TAA_PROJECT_ROOT"]).expanduser().resolve()

    if start is None:
        start = Path.cwd()

    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate

    raise FileNotFoundError("Could not find project root. Start Jupyter from the repo root.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "config"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / "genes.json") as f:
    GENE_CONFIG = json.load(f)

TARGET_GENES = GENE_CONFIG["taa_target_genes"]
CN_GENES = GENE_CONFIG["copy_number_feature_genes"]
MUTATION_GENES = GENE_CONFIG["mutation_feature_genes"]

print("Project root:", PROJECT_ROOT)
print("Target genes:", TARGET_GENES)

## Helper functions

In [ ]:
UUID_RE = re.compile(r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")
TCGA_BARCODE_RE = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})")


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace("-", "_", regex=False)
        .str.replace(".", "_", regex=False)
        .str.lower()
    )
    return df


def clean_tcga_barcode(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if "," in text:
        text = text.split(",")[0].strip()

    match = TCGA_BARCODE_RE.search(text)
    if match:
        return match.group(1)

    return pd.NA


def file_id_from_path(path: Path) -> str:
    for p in path.parents:
        if UUID_RE.match(p.name):
            return p.name
    return path.parent.name


def list_data_files(root: Path, suffixes: tuple[str, ...], exclude_logs=True):
    files = []
    if not root.exists():
        return files

    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if exclude_logs and "logs" in path.parts:
            continue
        if path.name.endswith(".partial") or path.name.endswith(".parcel"):
            continue
        if path.name.endswith(suffixes):
            files.append(path)

    return sorted(files)


def load_sample_sheet(modality: str) -> pd.DataFrame:
    folder = RAW_DATA_DIR / modality / "download_info"
    sample_sheets = sorted(folder.glob("gdc_sample_sheet*.tsv"))

    if not sample_sheets:
        raise FileNotFoundError(f"No gdc_sample_sheet*.tsv found in {folder}")

    df = pd.read_csv(sample_sheets[0], sep="\t", dtype=str)
    df = clean_columns(df)

    for col in ["file_id", "file_name", "case_id", "project_id", "sample_id", "sample_type", "case_submitter_id", "sample_submitter_id"]:
        if col not in df.columns:
            df[col] = pd.NA

    df["case_barcode"] = df["case_id"].map(clean_tcga_barcode)
    if df["case_barcode"].isna().all() and "case_submitter_id" in df.columns:
        df["case_barcode"] = df["case_submitter_id"].map(clean_tcga_barcode)

    return df


def get_meta_for_file(path: Path, sample_sheet: pd.DataFrame) -> dict:
    file_id = file_id_from_path(path)
    row = sample_sheet.loc[sample_sheet["file_id"] == file_id]

    if row.empty:
        return {
            "file_id": file_id,
            "case_barcode": pd.NA,
            "project_id": pd.NA,
            "sample_id": pd.NA,
            "sample_type": pd.NA,
            "file_name": path.name,
        }

    meta = row.iloc[0].to_dict()
    meta["file_id"] = file_id
    return meta


def read_gdc_tsv(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", comment="#", low_memory=False, **kwargs)


def summarize_and_save(df: pd.DataFrame, path: Path, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"{label}: {df.shape} -> {path}")

## RNA expression extraction

Extracts TAA RNA expression from GDC STAR-count files. Uses `tpm_unstranded` by default.

In [ ]:
def choose_expression_column(df: pd.DataFrame, preferred=None) -> str:
    if preferred and preferred in df.columns:
        return preferred

    for col in ["tpm_unstranded", "fpkm_uq_unstranded", "fpkm_unstranded", "unstranded"]:
        if col in df.columns:
            return col

    raise ValueError(f"Could not find expression column. Columns: {list(df.columns)[:40]}")


def extract_taa_rna(expression_column=None, limit=None):
    sample_sheet = load_sample_sheet("rna_expression")
    files = list_data_files(RAW_DATA_DIR / "rna_expression" / "files", (".tsv",))
    if limit:
        files = files[:limit]

    rows = []
    errors = []

    print(f"RNA files found: {len(files)}")

    for i, path in enumerate(files, start=1):
        try:
            meta = get_meta_for_file(path, sample_sheet)
            df = read_gdc_tsv(path)
            df.columns = [str(c).strip().lower() for c in df.columns]

            expr_col = choose_expression_column(df, expression_column)
            sub = df.loc[df["gene_name"].isin(TARGET_GENES), ["gene_name", expr_col]].copy()
            sub[expr_col] = pd.to_numeric(sub[expr_col], errors="coerce")

            values = dict(zip(sub["gene_name"], sub[expr_col]))
            row = {
                "file_id": meta.get("file_id"),
                "case_id": meta.get("case_barcode"),
                "project_id": meta.get("project_id"),
                "sample_id": meta.get("sample_id"),
                "sample_type": meta.get("sample_type"),
                "source_file": str(path),
                "expression_column": expr_col,
            }

            for gene in TARGET_GENES:
                raw = values.get(gene, np.nan)
                row[f"{gene}_rna_tpm"] = raw
                row[f"{gene}_rna_log2tpm"] = np.log2(raw + 1.0) if pd.notna(raw) else np.nan

            rows.append(row)

        except Exception as e:
            errors.append({"file": str(path), "error": str(e)})

        if i % 250 == 0:
            print(f"  processed {i}/{len(files)}")

    sample_level = pd.DataFrame(rows)
    target_cols = [f"{g}_rna_log2tpm" for g in TARGET_GENES]

    case_level = (
        sample_level[["case_id", "project_id"] + target_cols]
        .dropna(subset=["case_id"])
        .groupby("case_id", as_index=False)
        .agg({**{"project_id": "first"}, **{c: "mean" for c in target_cols}})
    )

    summarize_and_save(sample_level, PROCESSED_DATA_DIR / "rna_taa_sample_level.csv", "RNA sample level")
    summarize_and_save(case_level, PROCESSED_DATA_DIR / "rna_taa_case_level.csv", "RNA case level")

    if errors:
        summarize_and_save(pd.DataFrame(errors), PROCESSED_DATA_DIR / "rna_extract_errors.csv", "RNA errors")

    return sample_level, case_level

## Copy-number extraction

Extracts gene-level copy-number values for TAA genes and selected cancer driver genes.

In [ ]:
GENE_COL_CANDIDATES = ["gene_name", "gene", "gene_symbol", "symbol", "hugo_symbol", "gene_id"]


def detect_gene_col(df: pd.DataFrame) -> str:
    for col in GENE_COL_CANDIDATES:
        if col in df.columns:
            return col
    raise ValueError(f"Could not detect gene column. Columns: {list(df.columns)[:50]}")


def copy_number_value(df: pd.DataFrame) -> pd.Series:
    for col in [
        "copy_number",
        "total_copy_number",
        "integer_copy_number",
        "absolute_copy_number",
        "gene_level_copy_number",
        "modal_copy_number",
    ]:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce")

    if {"min_copy_number", "max_copy_number"}.issubset(df.columns):
        return (
            pd.to_numeric(df["min_copy_number"], errors="coerce") +
            pd.to_numeric(df["max_copy_number"], errors="coerce")
        ) / 2.0

    if "min_copy_number" in df.columns:
        return pd.to_numeric(df["min_copy_number"], errors="coerce")

    if "max_copy_number" in df.columns:
        return pd.to_numeric(df["max_copy_number"], errors="coerce")

    raise ValueError(f"Could not detect copy-number value column. Columns: {list(df.columns)[:80]}")


def extract_copy_number(limit=None):
    sample_sheet = load_sample_sheet("copy_number")
    files = list_data_files(RAW_DATA_DIR / "copy_number" / "files", (".tsv", ".txt"))
    if limit:
        files = files[:limit]

    rows = []
    errors = []

    print(f"Copy-number files found: {len(files)}")

    for i, path in enumerate(files, start=1):
        try:
            meta = get_meta_for_file(path, sample_sheet)
            df = read_gdc_tsv(path)
            df = clean_columns(df)

            gene_col = detect_gene_col(df)
            df["__cn_value__"] = copy_number_value(df)

            sub = df.loc[df[gene_col].astype(str).isin(CN_GENES), [gene_col, "__cn_value__"]]
            values = dict(zip(sub[gene_col].astype(str), sub["__cn_value__"]))

            row = {
                "file_id": meta.get("file_id"),
                "case_id": meta.get("case_barcode"),
                "project_id": meta.get("project_id"),
                "sample_id": meta.get("sample_id"),
                "sample_type": meta.get("sample_type"),
                "source_file": str(path),
            }

            for gene in CN_GENES:
                row[f"{gene}_cn"] = values.get(gene, np.nan)

            rows.append(row)

        except Exception as e:
            errors.append({"file": str(path), "error": str(e)})

        if i % 250 == 0:
            print(f"  processed {i}/{len(files)}")

    sample_level = pd.DataFrame(rows)
    cn_cols = [f"{g}_cn" for g in CN_GENES]

    case_level = (
        sample_level[["case_id", "project_id"] + cn_cols]
        .dropna(subset=["case_id"])
        .groupby("case_id", as_index=False)
        .agg({**{"project_id": "first"}, **{c: "mean" for c in cn_cols}})
    )

    summarize_and_save(sample_level, PROCESSED_DATA_DIR / "copy_number_sample_level.csv", "Copy-number sample level")
    summarize_and_save(case_level, PROCESSED_DATA_DIR / "copy_number_case_level.csv", "Copy-number case level")

    if errors:
        summarize_and_save(pd.DataFrame(errors), PROCESSED_DATA_DIR / "copy_number_extract_errors.csv", "Copy-number errors")

    return sample_level, case_level

## Mutation extraction

Creates binary mutation features for selected genes from masked somatic mutation MAF files.

In [ ]:
SILENT_OR_NONCODING = {
    "Silent",
    "Intron",
    "IGR",
    "RNA",
    "3'UTR",
    "5'UTR",
    "3'Flank",
    "5'Flank",
    "lincRNA",
}


def read_maf_minimal(path: Path) -> pd.DataFrame:
    needed = {"Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"}
    try:
        return pd.read_csv(
            path,
            sep="\t",
            comment="#",
            compression="gzip",
            low_memory=False,
            usecols=lambda c: c in needed,
        )
    except ValueError:
        df = pd.read_csv(path, sep="\t", comment="#", compression="gzip", low_memory=False)
        keep = [c for c in df.columns if c in needed]
        return df[keep]


def extract_mutations(nonsilent_only=True, limit=None):
    sample_sheet = load_sample_sheet("mutation")
    files = list_data_files(RAW_DATA_DIR / "mutation" / "files", (".maf.gz",))
    if limit:
        files = files[:limit]

    rows = []
    errors = []

    print(f"Mutation MAF.GZ files found: {len(files)}")

    for i, path in enumerate(files, start=1):
        try:
            meta = get_meta_for_file(path, sample_sheet)
            row = {
                "file_id": meta.get("file_id"),
                "case_id": meta.get("case_barcode"),
                "project_id": meta.get("project_id"),
                "sample_id": meta.get("sample_id"),
                "sample_type": meta.get("sample_type"),
                "source_file": str(path),
            }

            for gene in MUTATION_GENES:
                row[f"{gene}_mut"] = 0

            row["selected_driver_mut_count"] = 0
            row["maf_total_rows"] = 0

            df = read_maf_minimal(path)
            row["maf_total_rows"] = len(df)

            if not df.empty and "Hugo_Symbol" in df.columns:
                if nonsilent_only and "Variant_Classification" in df.columns:
                    df = df.loc[~df["Variant_Classification"].isin(SILENT_OR_NONCODING)]

                mutated = sorted(set(df.loc[df["Hugo_Symbol"].isin(MUTATION_GENES), "Hugo_Symbol"].dropna().astype(str)))

                for gene in mutated:
                    row[f"{gene}_mut"] = 1

                row["selected_driver_mut_count"] = len(mutated)

            rows.append(row)

        except Exception as e:
            errors.append({"file": str(path), "error": str(e)})

        if i % 500 == 0:
            print(f"  processed {i}/{len(files)}")

    sample_level = pd.DataFrame(rows)
    mut_cols = [f"{g}_mut" for g in MUTATION_GENES]

    agg = {c: "max" for c in mut_cols}
    agg.update({"project_id": "first", "selected_driver_mut_count": "sum", "maf_total_rows": "sum"})

    case_level = (
        sample_level[["case_id", "project_id", "selected_driver_mut_count", "maf_total_rows"] + mut_cols]
        .dropna(subset=["case_id"])
        .groupby("case_id", as_index=False)
        .agg(agg)
    )

    summarize_and_save(sample_level, PROCESSED_DATA_DIR / "mutation_sample_level.csv", "Mutation sample level")
    summarize_and_save(case_level, PROCESSED_DATA_DIR / "mutation_case_level.csv", "Mutation case level")

    if errors:
        summarize_and_save(pd.DataFrame(errors), PROCESSED_DATA_DIR / "mutation_extract_errors.csv", "Mutation errors")

    return sample_level, case_level

## Clinical feature extraction

In [ ]:
def read_clinical_tsv(path: Path) -> pd.DataFrame:
    return clean_columns(pd.read_csv(path, sep="\t", dtype=str, low_memory=False))


def normalize_clinical_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rename_map = {
        "cases_case_id": "clinical_case_uuid",
        "case_id": "clinical_case_uuid",
        "project_project_id": "project_id",
        "cases_submitter_id": "case_submitter_id",
        "cases_disease_type": "disease_type",
        "cases_primary_site": "primary_site",
        "demographic_age_at_index": "age_at_index",
        "demographic_days_to_birth": "days_to_birth",
        "demographic_days_to_death": "days_to_death",
        "demographic_ethnicity": "ethnicity",
        "demographic_race": "race",
        "demographic_sex_at_birth": "sex_at_birth",
        "demographic_gender": "gender",
        "demographic_vital_status": "vital_status",
        "demographic_year_of_birth": "year_of_birth",
        "diagnoses_age_at_diagnosis": "age_at_diagnosis",
        "diagnoses_ajcc_clinical_stage": "ajcc_clinical_stage",
        "diagnoses_ajcc_pathologic_stage": "ajcc_pathologic_stage",
        "diagnoses_tumor_grade": "tumor_grade",
        "diagnoses_morphology": "morphology",
        "diagnoses_tissue_or_organ_of_origin": "tissue_or_organ_of_origin",
        "diagnoses_prior_malignancy": "prior_malignancy",
        "diagnoses_prior_treatment": "prior_treatment",
        "diagnoses_classification_of_tumor": "classification_of_tumor",
        "diagnoses_last_known_disease_status": "last_known_disease_status",
        "diagnoses_progression_or_recurrence": "progression_or_recurrence",
        "diagnoses_year_of_diagnosis": "year_of_diagnosis",
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    return df


def normalize_sample_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rename_map = {
        "cases_case_id": "clinical_case_uuid",
        "case_id": "clinical_case_uuid",
        "samples_sample_id": "sample_id",
        "sample_id": "sample_id",
        "samples_submitter_id": "sample_submitter_id",
        "sample_submitter_id": "sample_submitter_id",
        "samples_sample_type": "sample_type",
        "sample_type": "sample_type",
        "samples_tissue_type": "tissue_type",
        "tissue_type": "tissue_type",
        "samples_tumor_descriptor": "tumor_descriptor",
        "tumor_descriptor": "tumor_descriptor",
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    return df


def first_existing(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]


def extract_clinical_features():
    raw_clin = RAW_DATA_DIR / "clinical"

    clinical_paths = sorted(raw_clin.rglob("clinical.tsv"))
    if not clinical_paths:
        raise FileNotFoundError(f"Could not find clinical.tsv under {raw_clin}")

    clinical = read_clinical_tsv(clinical_paths[0])
    clinical = normalize_clinical_columns(clinical)

    if "case_submitter_id" not in clinical.columns:
        raise ValueError(f"Clinical file missing case_submitter_id. Columns: {list(clinical.columns)[:80]}")

    clinical["case_id"] = clinical["case_submitter_id"].map(clean_tcga_barcode)

    keep_cols = [
        "case_id",
        "case_submitter_id",
        "clinical_case_uuid",
        "project_id",
        "primary_site",
        "disease_type",
        "age_at_index",
        "age_at_diagnosis",
        "days_to_birth",
        "sex_at_birth",
        "gender",
        "race",
        "ethnicity",
        "vital_status",
        "days_to_death",
        "year_of_birth",
        "year_of_diagnosis",
        "ajcc_pathologic_stage",
        "ajcc_clinical_stage",
        "tumor_grade",
        "morphology",
        "tissue_or_organ_of_origin",
        "prior_malignancy",
        "prior_treatment",
        "classification_of_tumor",
        "last_known_disease_status",
        "progression_or_recurrence",
    ]

    out = clinical[first_existing(clinical, keep_cols)].copy()

    sample_paths = sorted(raw_clin.rglob("sample.tsv"))
    if sample_paths:
        sample = read_clinical_tsv(sample_paths[0])
        sample = normalize_sample_columns(sample)

        # Some biospecimen files contain TCGA sample submitter IDs.
        if "sample_submitter_id" in sample.columns:
            sample["case_id"] = sample["sample_submitter_id"].map(clean_tcga_barcode)

        if "case_id" in sample.columns:
            sample_keep = first_existing(sample, ["case_id", "sample_id", "sample_submitter_id", "sample_type", "tissue_type", "tumor_descriptor"])
            sample = sample[sample_keep].copy()

            agg = {}
            for col in ["sample_type", "tissue_type", "tumor_descriptor"]:
                if col in sample.columns:
                    agg[col] = lambda x: "|".join(sorted(set(x.dropna().astype(str))))

            if agg:
                sample_agg = sample.groupby("case_id", as_index=False).agg(agg)
                out = out.merge(sample_agg, on="case_id", how="left")

    out = out.dropna(subset=["case_id"]).drop_duplicates(subset=["case_id"])

    summarize_and_save(out, PROCESSED_DATA_DIR / "clinical_features_case_level.csv", "Clinical case level")
    return out

## Build final modeling table

In [ ]:
def build_modeling_table():
    rna = pd.read_csv(PROCESSED_DATA_DIR / "rna_taa_case_level.csv", dtype={"case_id": str})
    cn = pd.read_csv(PROCESSED_DATA_DIR / "copy_number_case_level.csv", dtype={"case_id": str})
    mut = pd.read_csv(PROCESSED_DATA_DIR / "mutation_case_level.csv", dtype={"case_id": str})
    clinical = pd.read_csv(PROCESSED_DATA_DIR / "clinical_features_case_level.csv", dtype={"case_id": str})

    target_cols = [f"{g}_rna_log2tpm" for g in TARGET_GENES]
    cn_cols = [f"{g}_cn" for g in CN_GENES]
    mut_cols = [f"{g}_mut" for g in MUTATION_GENES]

    for df in [rna, cn, mut, clinical]:
        df["case_id"] = df["case_id"].map(clean_tcga_barcode)

    cn = cn.rename(columns={"project_id": "project_id_cn"})
    mut = mut.rename(columns={"project_id": "project_id_mut"})
    clinical = clinical.rename(columns={"project_id": "project_id_clinical"})

    table = rna.merge(cn, on="case_id", how="left")
    table = table.merge(mut, on="case_id", how="left")
    table = table.merge(clinical, on="case_id", how="left")

    for c in mut_cols:
        if c in table.columns:
            table[c] = pd.to_numeric(table[c], errors="coerce").fillna(0).astype(int)

    for c in ["selected_driver_mut_count", "maf_total_rows"]:
        if c in table.columns:
            table[c] = pd.to_numeric(table[c], errors="coerce").fillna(0)

    before = len(table)
    table = table.dropna(subset=target_cols)
    after = len(table)

    summarize_and_save(table, PROCESSED_DATA_DIR / "modeling_table.csv", "Final modeling table")

    summary = {
        "rna_cases": len(rna),
        "copy_number_cases": len(cn),
        "mutation_cases": len(mut),
        "clinical_cases": len(clinical),
        "modeling_rows_before_target_drop": before,
        "modeling_rows_after_target_drop": after,
        "rows_dropped_missing_targets": before - after,
        "target_columns": target_cols,
        "copy_number_columns": [c for c in cn_cols if c in table.columns],
        "mutation_columns": [c for c in mut_cols if c in table.columns],
        "copy_number_missing_fraction": {c: float(table[c].isna().mean()) for c in cn_cols if c in table.columns},
    }

    with open(PROCESSED_DATA_DIR / "modeling_table_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2)[:2000])
    return table, summary

## Run full processing

In [ ]:
# Set RUN_FULL_PROCESSING = False if you only want to test helper functions.
RUN_FULL_PROCESSING = True

if RUN_FULL_PROCESSING:
    rna_sample, rna_case = extract_taa_rna()
    cn_sample, cn_case = extract_copy_number()
    mut_sample, mut_case = extract_mutations()
    clinical = extract_clinical_features()
    modeling_table, modeling_summary = build_modeling_table()

    display(modeling_table.head())
else:
    print("RUN_FULL_PROCESSING is False. No data processing was executed.")